# check what folder we are on 

In [1]:
import os
os.getcwd()

'/Users/yerik/_apple_lib/_a_progs/_a2ms_env/_8_PROJECT_utils'

# 1 # ABLETON PROJECT FOLDER STRUCTURE 

In [12]:
# ---------------------------------------------------
# WRITE ADVANCED DATA LOADER
# ---------------------------------------------------

from pathlib import Path

code = '''
# =========================================================
# -----######-----######  CORE FUNCTION  -----######-----###
# =========================================================

import os
from datetime import datetime
from tqdm import tqdm

def _fld_1803_i1_GET_project_structure(base_dir, project_name):
    """
    Create a structured Ableton project folder.

    Parameters
    ----------
    base_dir : str
        Path where the project folder will be created
    project_name : str
        Name of the project folder

    Returns
    -------
    project_path : str
        Full path of the created project
    """

    # ---------- project root ----------
    project_path = os.path.join(base_dir, project_name)

    # ---------- folder structure ----------
    structure = [
        "_SAMPLES/drums",
        "_SAMPLES/percussion",
        "_SAMPLES/kicks",
        "_SAMPLES/loops",
        "_SAMPLES/synths",
        "_SAMPLES/fx",
        "_SAMPLES/vocals",
        "_RECORDED/resamples",
        "_RECORDED/recordings",
        "_EXPORTS/drafts",
        "_EXPORTS/final",
        "_REFERENCES/tracks",
        "_PRESETS/racks",
        "_PRESETS/fx_chains"
    ]

    # ---------- create root ----------
    os.makedirs(project_path, exist_ok=True)

    # ---------- create folders with TQDM ----------
    for folder in tqdm(structure, desc="Creating project structure"):
        full_path = os.path.join(project_path, folder)
        os.makedirs(full_path, exist_ok=True)

    # ---------- create placeholder ALS file ----------
    als_path = os.path.join(project_path, f"{project_name}.als")
    if not os.path.exists(als_path):
        open(als_path, 'a').close()

    # ---------- return ----------
    print(f"✅ Project created at:{project_path}")
    return project_path
'''

Path("src/make_folder_structure.py").write_text(code)

print("src/make_folder_structure.py")

src/make_folder_structure.py


# RENAME FILES IN FOLDER AND GET DF 

In [18]:
# ---------------------------------------------------
# WRITE ADVANCED DATA LOADER
# ---------------------------------------------------

from pathlib import Path

code = '''
# =========================================================
# -----######-----######  CORE FUNCTION  -----######-----###
# =========================================================

import os
import pandas as pd
from tqdm import tqdm

def _ren_1803_i5_GET_df_samples(samples_path):

    data = []
    counter_dict = {}

    for root, dirs, files in os.walk(samples_path):

        # 🔒 skip root _SAMPLES
        if root == samples_path:
            continue

        folder_name = os.path.basename(root)

        if folder_name not in counter_dict:
            counter_dict[folder_name] = 1

        for file in tqdm(files, desc=f"Processing {folder_name}"):

            # skip hidden/system
            if file.startswith('.') or file.startswith('._'):
                continue

            # 🔒 SKIP already renamed files
            if file.startswith(f"{folder_name}_"):
                continue

            old_path = os.path.join(root, file)

            if not os.path.isfile(old_path):
                continue

            name, ext = os.path.splitext(file)

            i = counter_dict[folder_name]

            new_name = f"{folder_name}_{i}_{name}{ext}"
            new_path = os.path.join(root, new_name)

            os.rename(old_path, new_path)

            file_size = os.path.getsize(new_path)

            data.append({
                "folder": folder_name,
                "file_name_old": file,
                "file_name_new": new_name,
                "name_old": name,
                "name_new": f"{folder_name}_{i}_{name}",
                "ext": ext.lower(),
                "Path_old": old_path,
                "Path": new_path,
                "file_size_bytes": file_size
            })

            counter_dict[folder_name] += 1

    df = pd.DataFrame(data)

    print(f"✅ Total files processed: {len(df)}")

    return df
'''

Path("src/rename_SAMPLE_files.py").write_text(code)

print("src/rename_SAMPLE_files.py")

src/rename_SAMPLE_files.py


# create module to perform EDA pt.1

In [9]:
# ---------------------------------------------------
# CREATE EDA MODULE
# ---------------------------------------------------

from pathlib import Path

code = '''
# ---------------------------------------------------------
# SIMPLE EDA (STEPS 1–6) + REPORT EXPORT
# does NOT modify df
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path

def eda_01_06_GET_report(df, path_out="reports/tables/eda_summary.csv"):

    # ------------------------------------------------
    # Console overview
    # ------------------------------------------------
    print("\\nDATA SHAPE:", df.shape)
    print("\\nCOLUMNS:", list(df.columns))

    print("\\nHEAD:\\n", df.head())
    print("\\nTAIL:\\n", df.tail())

    print("\\nDATA INFO:")
    df.info()

    print("\\nDUPLICATE ROWS:", df.duplicated().sum())

    # ------------------------------------------------
    # Target overview
    # ------------------------------------------------
    if "purchase" in df.columns:
        print("\\nTARGET DISTRIBUTION:")
        print(df["purchase"].value_counts())

    # ------------------------------------------------
    # Report table
    # ------------------------------------------------
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.values,
        "missing": df.isna().sum().values,
        "missing_%": (df.isna().mean()*100).round(2).values,
        "unique": df.nunique().values
    })

    # ------------------------------------------------
    # Save report
    # ------------------------------------------------
    Path(path_out).parent.mkdir(parents=True, exist_ok=True)
    report.to_csv(path_out, index=False)

    print("\\nREPORT SAVED →", path_out)

    return report
'''

Path("src/eda.py").write_text(code)

print("Created: src/eda.py")

Created: src/eda.py


# Preprocess data , x , y variable 

In [10]:
from pathlib import Path

code = '''
# ---------------------------------------------------------
# PREPROCESSING
# encode categoricals + split features/target
# ---------------------------------------------------------

import pandas as pd

def prep_features_target(df, target="purchase"):

    # separate X and y
    X = df.drop(columns=[target])
    y = df[target]

    # encode categorical variables
    X = pd.get_dummies(X, drop_first=True)

    print("\\nFEATURE MATRIX:", X.shape)
    print("TARGET:", y.shape)

    return X, y
'''

Path("src/preprocessing.py").write_text(code)

print("Created: src/preprocessing.py")

Created: src/preprocessing.py


# model Preparation 

In [12]:
from pathlib import Path

code = '''
# ---------------------------------------------------------
# MODEL PREP
# train / test split
# ---------------------------------------------------------

from sklearn.model_selection import train_test_split

def split_train_test(X, y, test_size=0.2, random_state=42):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state
    )

    print("\\nTRAIN:", X_train.shape)
    print("TEST:", X_test.shape)

    return X_train, X_test, y_train, y_test
'''

Path("src/modeling.py").write_text(code)

print("Created: src/modeling.py")

Created: src/modeling.py
